# SemIf / Qwen3.5-4B direct logit readout on Colab L4

This notebook is a thin reproducible wrapper around `scripts/run_experiment.py`. It measures model loading, first inference, warmup, steady-state throughput, and CUDA peak memory. It uses the pinned TheoLeeCJ/SemIf implementation and does not use the separate AlexWortega NLI head.

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_experiment.py').exists():
    candidate = Path('/content/jev-colab-lab/experiments/semif')
    if (candidate / 'scripts' / 'run_experiment.py').exists():
        ROOT = candidate
assert (ROOT / 'scripts' / 'run_experiment.py').exists(), f'Open this notebook from {ROOT}'
print(ROOT)

In [ ]:
subprocess.run(['uv', 'sync', '--extra', 'gpu', '--group', 'dev', '--frozen'], cwd=ROOT, check=True)
subprocess.run(['uv', 'pip', 'uninstall', '--system', 'torchvision', 'torchaudio', 'librosa'], check=False)
subprocess.run(['uv', 'run', '--extra', 'gpu', 'python', 'scripts/validate_fixture.py', 'data/questions.jsonl'], cwd=ROOT, check=True)

In [ ]:
output = ROOT / 'results' / 'semif-l4-notebook.json'
subprocess.run([
    'uv', 'run', '--extra', 'gpu', 'python', 'scripts/run_experiment.py',
    '--input', 'data/questions.jsonl', '--output', str(output),
    '--expected-gpu', 'L4', '--warmup', '2', '--repeats', '5',
], cwd=ROOT, check=True)
print(output)

In [ ]:
import json
result = json.loads(output.read_text(encoding='utf-8'))
print(json.dumps({
    'status': result['status'],
    'gpu': result.get('runtime', {}).get('gpu'),
    'load_seconds': result.get('metrics', {}).get('model_load_seconds'),
    'first_inference_seconds': result.get('metrics', {}).get('first_inference_wall_seconds'),
    'steady_state': result.get('metrics', {}).get('steady_state'),
    'peak_vram': result.get('metrics', {}).get('peak_vram'),
}, ensure_ascii=False, indent=2))

## Fixed JevDash model-only follow-up

The following optional cells use the same pinned SemIf/Qwen revision against the unmodified JevDash checkout. They run `uv` in subprocesses so the notebook does not silently switch the active kernel environment. The audit is input/order evidence; the episode runner has no mock, fallback, or helper override path.

In [ ]:
GAME_REPOSITORY = 'https://github.com/Sunwood-ai-labs/jevdash'
GAME_COMMIT = 'eb2f92617bab5d5021a5e3cf5ef2bdaf8207d480'
JEVDASH_ROOT = Path('/content/jevdash')
GAME_COMMIT_FILE = Path('/content/jevdash-commit.txt')
if not JEVDASH_ROOT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', GAME_REPOSITORY, str(JEVDASH_ROOT)], check=True)
subprocess.run(['git', '-C', str(JEVDASH_ROOT), 'checkout', '--detach', GAME_COMMIT], check=True)
GAME_COMMIT_FILE.write_text(GAME_COMMIT + '\n', encoding='utf-8')
print(JEVDASH_ROOT, GAME_COMMIT)

In [ ]:
audit_output = ROOT / 'results' / 'semif-jevdash-input-audit-notebook.json'
subprocess.run([
    'uv', 'run', '--extra', 'gpu', 'python', 'scripts/jevdash_audit.py',
    '--game-root', str(JEVDASH_ROOT),
    '--game-commit-file', str(GAME_COMMIT_FILE),
    '--output', str(audit_output),
], cwd=ROOT, check=True)
print(audit_output)

In [ ]:
episode_output = ROOT / 'results' / 'semif-jevdash-episode-notebook.json'
video_output = ROOT / 'results' / 'semif-jevdash-notebook.mp4'
subprocess.run([
    'uv', 'run', '--extra', 'gpu', 'python', 'scripts/jevdash_play.py',
    '--game-root', str(JEVDASH_ROOT),
    '--game-commit-file', str(GAME_COMMIT_FILE),
    '--output-video', str(video_output),
    '--output-json', str(episode_output),
    '--level', '1', '--seed', '42', '--fps', '60',
    '--frames-per-decision', '8', '--max-frames', '1800',
    '--static-terminal-frames', '120',
], cwd=ROOT, check=True)
print(episode_output, video_output)

In [ ]:
episode = json.loads(episode_output.read_text(encoding='utf-8'))
assert episode['status'] == 'completed'
assert episode['runtime']['gpu']['name'].endswith('L4')
assert episode['source']['mock_agent_used'] is False
assert episode['summary']['control_mode'] == 'model_only'
print(json.dumps({
    'outcome': episode['outcome'],
    'gpu': episode['runtime']['gpu']['name'],
    'controller_prompt_version': episode['source']['controller_prompt_version'],
    'summary': episode['summary'],
}, ensure_ascii=False, indent=2))